# Whakaari data construction

This notebook builds the canonical hourly Whakaari dataset used by the Cause–Trigger analysis. It retains seven variables: four waveform-derived observables, rainfall, pressure drop, and the lagged GNSS deformation-rate proxy.

Short waveform gaps of at most two seconds are interpolated. Longer waveform gaps are not imputed; affected hourly rows are excluded from the complete-case canonical dataset and reported explicitly.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/whakaari"
sys.path.append(str(SRC_DIR))

from whakaari_config import (
    TILDE_DATA_URL,
    WHAKAARI_ANALYSIS_COLUMNS,
    WHAKAARI_END,
    WHAKAARI_ERUPTION_TIME,
    WHAKAARI_LAT,
    WHAKAARI_LON,
    WHAKAARI_START,
    WHAKAARI_WAVEFORM_CONFIG,
    whakaari_observable_metadata,
)
from whakaari_dataset import (
    build_master_dataframe,
    prepare_analysis_dataframe,
    preprocessing_report,
    save_whakaari_analysis_dataset,
)
from whakaari_geonet import (
    load_gnss_deformation,
    load_weather_vars,
)
from whakaari_plotting_utils import (
    dataset_health_report,
    distribution_summary,
    plot_whakaari_all_variables_map,
    plot_whakaari_loglog_distributions,
    plot_whakaari_thesis_figures,
)
from whakaari_waveform import build_waveform_dataset

BUFFER_START = (
    pd.Timestamp(WHAKAARI_START, tz="UTC") - pd.Timedelta(days=2)
).strftime("%Y-%m-%d")

## External variables

GNSS displacement is downloaded for RGWC and RGWI and converted to their vertical-displacement difference. The past-only daily change is constructed later. Weather data are cached and include hourly precipitation and pressure drop.

In [ ]:
gnss = load_gnss_deformation(
    tilde_data_base_url=TILDE_DATA_URL,
    start=BUFFER_START,
    end=WHAKAARI_END,
    cache_path="../data/whakaari/whakaari_gnss_deformation.csv",
    redownload=True,
)

weather_vars = load_weather_vars(
    lat=WHAKAARI_LAT,
    lon=WHAKAARI_LON,
    start=BUFFER_START,
    end=WHAKAARI_END,
    cache_path="../data/whakaari/whakaari_openmeteo_weather.csv",
    redownload=True,
)

display(dataset_health_report(gnss, "GNSS deformation level"))
display(dataset_health_report(weather_vars, "Open-Meteo variables"))

## Waveform features

WSRZ HHZ data are processed into hydrothermal RMS, past-smoothed spectral contrast, STA/LTA event rate, and the 5–15 Hz tremor-response anomaly. The cache also preserves any day-level extraction failures.

In [ ]:
waveform_df, waveform_failures = build_waveform_dataset(
    client=Client("GEONET"),
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    cfg=WHAKAARI_WAVEFORM_CONFIG,
    save_path="../data/whakaari/whakaari_waveform.pkl",
    overwrite=True, # True to redownload, False to load cache
)

display(dataset_health_report(waveform_df, "WSRZ waveform features"))
display(pd.DataFrame(waveform_failures, columns=["date", "error"]))

## Save hourly dataset

All sources are aligned to the same hourly grid. The GNSS value used during day \(D\) is the deformation change from \(D-2\) to \(D-1\), so it is past-only. Rows unresolved in any retained variable are removed without interpolating long waveform gaps.

In [ ]:
whakaari_master = build_master_dataframe(
    wave=waveform_df,
    weather_vars=weather_vars,
    gnss=gnss,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
)

whakaari_dataset = prepare_analysis_dataframe(
    whakaari_master,
    required_columns=WHAKAARI_ANALYSIS_COLUMNS,
)

construction = preprocessing_report(
    rawest_df=whakaari_master,
    prepared_df=whakaari_dataset,
)

dataset_path = save_whakaari_analysis_dataset(
    whakaari_dataset,
    output_dir="../data/whakaari",
)

display(construction["summary"])
display(construction["missing_by_variable"])
display(construction["dropped_ranges"])
display(dataset_health_report(whakaari_dataset, "Canonical Whakaari dataset"))
display(distribution_summary(whakaari_dataset, "Canonical Whakaari dataset"))

print(f"Saved: {dataset_path}")

The expected dropped ranges are the initial proxy warm-up and the documented WSRZ waveform outage around 18–19 November 2019. The outage is retained as missing time rather than filled.

### Log-log distribution diagnostics

These diagnostics use all seven canonical unscaled variables, including GNSS. True logarithmic axes require positive values, so zeros and negative observations are excluded variable by variable and reported explicitly.

In [ ]:
_, _, whakaari_loglog_report = plot_whakaari_loglog_distributions(
    dataframe=whakaari_dataset,
    save_dir="figures",
)

display(whakaari_loglog_report)

### Final overview figures

In [ ]:
plot_whakaari_thesis_figures(
    csv_path=dataset_path,
    eruption_time=WHAKAARI_ERUPTION_TIME,
    save_dir="figures",
    include_titles=False,
)

### Geographic Visualization

In [ ]:
metadata = whakaari_observable_metadata()

fig, ax, variable_table = plot_whakaari_all_variables_map(
    metadata=metadata,
    satellite=True,
    save_dir="figures",
    filename="whakaari_map",
)

display(variable_table)